# Getting Financial Data - Pandas Datareader

### Introduction:

This time you will get data from a website.


### Step 1. Import the necessary libraries

In [90]:
import pandas as pd
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder\
                    .appName('')\
                    .getOrCreate()


In [91]:
os.getcwd()

'/tf/pyspark_exercises/09_Time_Series/Getting_Financial_Data'

In [92]:
os.chdir('/tf/pyspark_exercises/09_Time_Series/Getting_Financial_Data')

### Step 2. Create your time range (start and end variables). The start date should be 01/01/2015 and the end should today (whatever your today is).

In [93]:
from datetime import datetime

hoy = datetime.now().date()

start_date = datetime.strptime('01/01/2015', '%m/%d/%Y').date()

In [94]:
serie_indice = pd.date_range(start=start_date,
              end=hoy)

serie_indice

DatetimeIndex(['2015-01-01', '2015-01-02', '2015-01-03', '2015-01-04',
               '2015-01-05', '2015-01-06', '2015-01-07', '2015-01-08',
               '2015-01-09', '2015-01-10',
               ...
               '2025-07-06', '2025-07-07', '2025-07-08', '2025-07-09',
               '2025-07-10', '2025-07-11', '2025-07-12', '2025-07-13',
               '2025-07-14', '2025-07-15'],
              dtype='datetime64[ns]', length=3849, freq='D')

### Step 3. Get an API key for one of the APIs that are supported by Pandas Datareader, preferably for AlphaVantage.

If you do not have an API key for any of the supported APIs, it is easiest to get one for [AlphaVantage](https://www.alphavantage.co/support/#api-key). (Note that the API key is shown directly after the signup. You do *not* receive it via e-mail.)

(For a full list of the APIs that are supported by Pandas Datareader, [see here](https://pydata.github.io/pandas-datareader/readers/index.html). As the APIs are provided by third parties, this list may change.)

In [95]:
!pip install dotenv



[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [96]:
from dotenv import load_dotenv

load_dotenv()
CLAVE = 'API_KEY'
API_KEY = os.getenv(CLAVE)



### Step 4. Use Pandas Datarader to read the daily time series for the Apple stock (ticker symbol AAPL) between 01/01/2015 and today, assign it to df_apple and print it.

In [97]:
!pip install pandas_datareader



[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [98]:
import pandas_datareader.data as web
from datetime import datetime

start_date = datetime(2015, 1, 1)
end_date = datetime.today()

# Descargar datos usando Alpha Vantage
df_apple = web.DataReader('AAPL', 'av-daily', start=start_date, end=end_date, api_key=API_KEY)

# Mostrar resultados
print(df_apple.head())

              open    high      low   close    volume
2015-01-02  111.39  111.44  107.350  109.33  53204626
2015-01-05  108.29  108.65  105.410  106.25  64285491
2015-01-06  106.54  107.43  104.630  106.26  65797116
2015-01-07  107.20  108.20  106.695  107.75  40105934
2015-01-08  109.23  112.15  108.700  111.89  59364547


In [99]:
apple = spark.createDataFrame(df_apple)
apple.show()

+-------+--------+-------+------+---------+
|   open|    high|    low| close|   volume|
+-------+--------+-------+------+---------+
| 111.39|  111.44| 107.35|109.33| 53204626|
| 108.29|  108.65| 105.41|106.25| 64285491|
| 106.54|  107.43| 104.63|106.26| 65797116|
|  107.2|   108.2|106.695|107.75| 40105934|
| 109.23|  112.15|  108.7|111.89| 59364547|
| 112.67|  113.25| 110.21|112.01| 53315099|
|  112.6|  112.63|  108.8|109.25| 49650790|
| 111.43|   112.8| 108.91|110.22| 67091928|
| 109.04|  110.49|  108.5| 109.8| 48956588|
|  110.0|  110.06| 106.66|106.82| 60013996|
| 107.03|  107.58|  105.2|105.99| 78513345|
| 107.84|108.9667|  106.5|108.72| 49899907|
| 108.95|  111.06| 108.27|109.55| 48575897|
| 110.26|  112.47| 109.72| 112.4| 53796409|
|  112.3|  113.75| 111.53|112.98| 46464828|
| 113.74|114.3626|  112.8| 113.1| 55614979|
| 112.42|  112.48| 109.03|109.14| 95568749|
|117.625|  118.12| 115.31|115.31|146477063|
| 116.32|  119.19| 115.56| 118.9| 84436432|
|  118.4|   120.0| 116.85|117.16

### Step 5. Add a new column "stock" to the dataframe and add the ticker symbol

In [100]:
apple = apple.withColumn('stock', F.lit('AAPL'))
apple.show()

+-------+--------+-------+------+---------+-----+
|   open|    high|    low| close|   volume|stock|
+-------+--------+-------+------+---------+-----+
| 111.39|  111.44| 107.35|109.33| 53204626| AAPL|
| 108.29|  108.65| 105.41|106.25| 64285491| AAPL|
| 106.54|  107.43| 104.63|106.26| 65797116| AAPL|
|  107.2|   108.2|106.695|107.75| 40105934| AAPL|
| 109.23|  112.15|  108.7|111.89| 59364547| AAPL|
| 112.67|  113.25| 110.21|112.01| 53315099| AAPL|
|  112.6|  112.63|  108.8|109.25| 49650790| AAPL|
| 111.43|   112.8| 108.91|110.22| 67091928| AAPL|
| 109.04|  110.49|  108.5| 109.8| 48956588| AAPL|
|  110.0|  110.06| 106.66|106.82| 60013996| AAPL|
| 107.03|  107.58|  105.2|105.99| 78513345| AAPL|
| 107.84|108.9667|  106.5|108.72| 49899907| AAPL|
| 108.95|  111.06| 108.27|109.55| 48575897| AAPL|
| 110.26|  112.47| 109.72| 112.4| 53796409| AAPL|
|  112.3|  113.75| 111.53|112.98| 46464828| AAPL|
| 113.74|114.3626|  112.8| 113.1| 55614979| AAPL|
| 112.42|  112.48| 109.03|109.14| 95568749| AAPL|


In [101]:
df_apple['stock'] = 'AAPL'

In [102]:
df_apple

,open,high,low,close,volume,stock
2015-01-02,111.390,111.44,107.350,109.33,53204626,AAPL
2015-01-05,108.290,108.65,105.410,106.25,64285491,AAPL
2015-01-06,106.540,107.43,104.630,106.26,65797116,AAPL
2015-01-07,107.200,108.20,106.695,107.75,40105934,AAPL
2015-01-08,109.230,112.15,108.700,111.89,59364547,AAPL
...,...,...,...,...,...,...
2025-07-08,210.100,211.43,208.450,210.01,42848928,AAPL
2025-07-09,209.530,211.33,207.220,211.14,48749367,AAPL
2025-07-10,210.505,213.48,210.030,212.41,44443635,AAPL
2025-07-11,210.565,212.13,209.860,211.16,39765812,AAPL


### Step 6. Repeat the two previous steps for a few other stocks, always creating a new dataframe: Tesla, IBM and Microsoft. (Ticker symbols TSLA, IBM and MSFT.)

In [103]:

nuevos_datos = ['TSLA', 'IBM', 'MSFT']


def create_data(lista):
    dfs = []
    for dato in lista:
        df = web.DataReader(dato, 'av-daily', start_date, end_date, api_key=API_KEY)
        df['stock'] = dato
        dfs.append(df)
    
    return pd.concat(dfs)
    

df = create_data(nuevos_datos)

In [104]:
df

,open,high,low,close,volume,stock
2015-01-02,222.870,223.2500,213.2600,219.310,4764443,TSLA
2015-01-05,214.550,216.5000,207.1626,210.090,5368477,TSLA
2015-01-06,210.060,214.2000,204.2100,211.280,6261936,TSLA
2015-01-07,213.350,214.7800,209.7800,210.950,2968390,TSLA
2015-01-08,212.810,213.7999,210.0100,210.615,3442509,TSLA
...,...,...,...,...,...,...
2025-07-08,497.240,498.2000,494.1100,496.620,11846586,MSFT
2025-07-09,500.300,506.7800,499.7400,503.510,18659538,MSFT
2025-07-10,503.050,504.4400,497.7500,501.480,16498740,MSFT
2025-07-11,498.470,505.0300,497.7950,503.320,16459512,MSFT


In [105]:
df_rest = spark.createDataFrame(df)

df_rest.show()

+------+--------+--------+-------+--------+-----+
|  open|    high|     low|  close|  volume|stock|
+------+--------+--------+-------+--------+-----+
|222.87|  223.25|  213.26| 219.31| 4764443| TSLA|
|214.55|   216.5|207.1626| 210.09| 5368477| TSLA|
|210.06|   214.2|  204.21| 211.28| 6261936| TSLA|
|213.35|  214.78|  209.78| 210.95| 2968390| TSLA|
|212.81|213.7999|  210.01|210.615| 3442509| TSLA|
|208.92|  209.98|  204.96| 206.66| 4580722| TSLA|
|203.05|  204.47|  199.25| 202.21| 5950280| TSLA|
|203.32|  207.61| 200.911| 204.25| 4477320| TSLA|
|185.83|   195.2|   185.0| 192.69|11551855| TSLA|
|194.49|195.7499|   190.0| 191.87| 5216524| TSLA|
| 190.7|  194.49|  189.65| 193.07| 3603158| TSLA|
|193.87|194.1199|  187.04| 191.93| 4503182| TSLA|
|189.55|  198.68|  189.51| 196.57| 4153043| TSLA|
| 197.0|  203.24|   195.2| 201.62| 4116905| TSLA|
|200.29|   203.5|  198.33| 201.29| 3442371| TSLA|
|201.83|  208.62|  201.05| 206.55| 3234522| TSLA|
|204.42|  208.03|   203.3| 205.98| 2781024| TSLA|


### Step 7. Combine the four separate dataFrames into one combined dataFrame df that holds the information for all four stocks

In [106]:
df_total = apple.union(df_rest)


In [107]:
df_total.tail(10)

[Row(open=497.04, high=500.76, low=495.33, close=497.41, volume=28368991, stock='MSFT'),
 Row(open=496.47, high=498.05, low=490.98, close=492.05, volume=19945375, stock='MSFT'),
 Row(open=489.99, high=493.5, low=488.7, close=491.09, volume=16319641, stock='MSFT'),
 Row(open=493.81, high=500.13, low=493.44, close=498.84, volume=13984829, stock='MSFT'),
 Row(open=497.38, high=498.75, low=495.225, close=497.72, volume=13981605, stock='MSFT'),
 Row(open=497.24, high=498.2, low=494.11, close=496.62, volume=11846586, stock='MSFT'),
 Row(open=500.3, high=506.78, low=499.74, close=503.51, volume=18659538, stock='MSFT'),
 Row(open=503.05, high=504.44, low=497.75, close=501.48, volume=16498740, stock='MSFT'),
 Row(open=498.47, high=505.03, low=497.795, close=503.32, volume=16459512, stock='MSFT'),
 Row(open=501.515, high=503.97, low=501.03, close=503.02, volume=12058848, stock='MSFT')]

In [108]:
df_all = pd.concat([df_apple,
                     df])

df_all

,open,high,low,close,volume,stock
2015-01-02,111.390,111.44,107.350,109.33,53204626,AAPL
2015-01-05,108.290,108.65,105.410,106.25,64285491,AAPL
2015-01-06,106.540,107.43,104.630,106.26,65797116,AAPL
2015-01-07,107.200,108.20,106.695,107.75,40105934,AAPL
2015-01-08,109.230,112.15,108.700,111.89,59364547,AAPL
...,...,...,...,...,...,...
2025-07-08,497.240,498.20,494.110,496.62,11846586,MSFT
2025-07-09,500.300,506.78,499.740,503.51,18659538,MSFT
2025-07-10,503.050,504.44,497.750,501.48,16498740,MSFT
2025-07-11,498.470,505.03,497.795,503.32,16459512,MSFT


### Step 8. Shift the stock column into the index (making it a multi-level index consisting of the ticker symbol and the date).

In [109]:
df_all = df_all.reset_index().rename(columns={'index':'date'}).set_index(['stock', 'date'])
df_all

open    high      low   close    volume
stock date                                                  
AAPL  2015-01-02  111.390  111.44  107.350  109.33  53204626
      2015-01-05  108.290  108.65  105.410  106.25  64285491
      2015-01-06  106.540  107.43  104.630  106.26  65797116
      2015-01-07  107.200  108.20  106.695  107.75  40105934
      2015-01-08  109.230  112.15  108.700  111.89  59364547
...                   ...     ...      ...     ...       ...
MSFT  2025-07-08  497.240  498.20  494.110  496.62  11846586
      2025-07-09  500.300  506.78  499.740  503.51  18659538
      2025-07-10  503.050  504.44  497.750  501.48  16498740
      2025-07-11  498.470  505.03  497.795  503.32  16459512
      2025-07-14  501.515  503.97  501.030  503.02  12058848

[10588 rows x 5 columns]

### Step 7. Create a dataFrame called vol, with the volume values.

In [110]:
volume = df_all['volume'].to_frame()
volume

volume
stock date                
AAPL  2015-01-02  53204626
      2015-01-05  64285491
      2015-01-06  65797116
      2015-01-07  40105934
      2015-01-08  59364547
...                    ...
MSFT  2025-07-08  11846586
      2025-07-09  18659538
      2025-07-10  16498740
      2025-07-11  16459512
      2025-07-14  12058848

[10588 rows x 1 columns]

### Step 8. Aggregate the data of volume to weekly.
Hint: Be careful to not sum data from the same week of 2015 and other years.

In [111]:
volume = volume.reset_index()
volume['date'] = pd.to_datetime(volume['date'])


volume['year'] = volume['date'].dt.year
volume['week'] = volume['date'].dt.weekday



volume.pivot_table(values='volume',
                   columns='stock',
                   index=['year', 'week'],
                   aggfunc='sum')


stock            AAPL        IBM        MSFT        TSLA
year week                                               
2015 0     2415238882  215979965  1631909915   223329061
     1     2787038692  247110083  1905021157   223809172
     2     2798562132  214572756  1796296397   225518852
     3     2477651236  205003217  1687277329   217666303
     4     2585825833  222879500  2037077513   196384992
2016 0     1653803861  176302452  1352100063   206905947
     1     1919586292  208449347  1495587864   231845229
     2     2169128908  199797154  1575433057   254535281
     3     1970448080  215845036  1508113659   241440305
     4     1972904644  218700804  1888705345   227481535
2017 0     1198412023  185537122   973157162   304215326
     1     1369231889  208018284  1027778051   318370994
     2     1485810141  248214777  1080290301   322686977
     3     1251521545  201912355  1148793912   339793211
     4     1382237225  215024872  1290151801   288402099
2018 0     1609513251  238504921  1434140674   422920790
     1     1580499534  256698867  1518345096   451176912
     2     1645502308  289773823  1555407608   410950268
     3     1682428122  283482993  1586664583   431137471
     4     1949150083  291968089  1750687581   440651164
2019 0     1329853908  163853829  1093039731   446697600
     1     1457607673  194767786  1227434663   424183893
     2     1402118170  199516396  1220217648   440664100
     3     1378535120  199929934  1288910763   502026397
     4     1518453282  190739350  1385499290   502931512
2020 0     3712981893  264566749  1824017371  1460752677
     1     4011924827  290628991  1958041213  1578981561
     2     3500221911  279196319  1895444852  1524920458
     3     3465155314  282743972  1933182477  1508772762
     4     3862299000  285297642  1914569536  1467588541
2021 0     4262219230  244518772  1194711123  1426481768
     1     4701746853  275152345  1350857640  1485848068
     2     4601679120  253228284  1381014648  1348810022
     3     4570824433  274609931  1290214544  1312724306
     4     4661878484  302525868  1332636776  1321396613
2022 0     3923882129  211081360  1371870447  2162025496
     1     4318027113  287715708  1575256303  2660587868
     2     4381862572  243099064  1653894498  2583447915
     3     4587382137  258845280  1606022209  2671191744
     4     4839038182  255963423  1621830546  2603895538
2023 0     2531914614  175571363  1104283218  6031247249
     1     2792991166  200044383  1347630457  6757969946
     2     3059193052  224548074  1527822856  7284751540
     3     3070217152  243300292  1446051381  7074130201
     4     3350573142  269447854  1493669990  7191126772
2024 0     2702047854  178954063   897557674  4708947353
     1     2802326005  192458215  1027465028  4945632730
     2     2616312878  205598193  1008813113  4669455899
     3     2832653992  250669817  1085470991  4810360275
     4     3435349739  231235270  1164355766  4743154969
2025 0     1537694878  110823971   580361250  2786846518
     1     1472495345  110541648   552108225  3016366862
     2     1461262216  113317605   573616280  2976286527
     3     1398092719  125082964   605196473  2734068814
     4     1658025387  129836555   656427770  2779488117

### Step 9. Find all the volume traded in the year of 2015

In [114]:
volume = volume.pivot_table(values='volume',
                   columns='stock',
                   index=['year', 'week'],
                   aggfunc='sum')

In [124]:
volume.loc[(2015), :].sum(axis=0).to_frame('Total_volumen_2025')

,Total_volumen_2025
stock,
AAPL,13064316775
IBM,1105545521
MSFT,9057582311
TSLA,1086708380
